# DiD-BCF — B1_baseline (linearity_degree=2)

**Workstream B1 · canonical DiD (selection on unobservables)**

moderate unit FE, selection on unobservables, iid errors

Fits DiD-BCF on the `B1_baseline` scenario at `linearity_degree=2` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.0 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_baseline",
    linearity_degree=2,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_baseline_lin_2] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_baseline: 100%|██████████| 100/100 [6:09:03<00:00, 221.43s/fit]

[B1_baseline_lin_2] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_baseline_lin_2.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_baseline,2,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.286667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
1,canonical,B1_baseline,2,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.293333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
2,canonical,B1_baseline,2,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.448000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
3,canonical,B1_baseline,2,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.468000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563
4,canonical,B1_baseline,2,200,0,ES,k=0,NaN,NaN,0.0,...,0.286667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.848563


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_baseline,2,200,ATT,ATT,power,-0.607316,-3.745341,0.31,...,0.759624,1.458279,0.652672,3.745341,1.0,0.0,0.813395,3.746895,0.354065,5.627052
1,canonical,B1_baseline,2,200,ES,k=0,power,-0.615617,-3.754080,0.37,...,0.933405,1.329482,0.672553,3.754080,1.0,0.0,0.827252,3.755257,0.426323,33.851839
2,canonical,B1_baseline,2,200,ES,k=1,power,-0.603711,-3.746435,0.37,...,0.916984,1.509096,0.642549,3.746435,1.0,0.0,0.803716,3.748292,0.436730,5.037632
3,canonical,B1_baseline,2,200,ES,k=2,power,-0.606119,-3.740907,0.39,...,0.945447,1.638447,0.661985,3.740907,1.0,0.0,0.828055,3.742929,0.422156,4.328474
4,canonical,B1_baseline,2,200,ES,k=3,power,-0.603818,-3.739940,0.38,...,0.929259,1.754681,0.656346,3.739940,1.0,0.0,0.815847,3.742312,0.428290,4.131400
5,canonical,B1_baseline,2,200,GATT,g=4_t=4,power,-0.615617,-3.754080,0.37,...,0.933405,1.329482,0.672553,3.754080,1.0,0.0,0.827252,3.755257,0.426323,33.851839
6,canonical,B1_baseline,2,200,GATT,g=4_t=5,power,-0.603711,-3.746435,0.37,...,0.916984,1.509096,0.642549,3.746435,1.0,0.0,0.803716,3.748292,0.436730,5.037632
7,canonical,B1_baseline,2,200,GATT,g=4_t=6,power,-0.606119,-3.740907,0.39,...,0.945447,1.638447,0.661985,3.740907,1.0,0.0,0.828055,3.742929,0.422156,4.328474
8,canonical,B1_baseline,2,200,GATT,g=4_t=7,power,-0.603818,-3.739940,0.38,...,0.929259,1.754681,0.656346,3.739940,1.0,0.0,0.815847,3.742312,0.428290,4.131400


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_baseline,2,200,corrected,100,401.24,1.160747,0.304109,0.978541,...,0.251925,0.056131,0.237855,0.067788,0.281847,0.078589,0.776651,0.084035,0.931274,0.106734
1,canonical,B1_baseline,2,200,plain,100,401.24,3.846793,0.106058,3.745341,...,0.996420,0.017403,0.000000,0.000000,0.000000,0.000000,1.240269,0.064068,1.563819,0.080079
